# U-Net Speech Denoiser — Colab Master

**Architecture**: AdvancedUNetSE with LFBypass + LF-enhanced skip connections  
**Dataset**: 70% LF noise (HVAC/fan/hum) / 30% HF transients  
**Target**: Thai speech denoising, 16 kHz, 2-second segments  

Run cells **top-to-bottom** on first launch.  
On resume, cells 1–4 must still run; cell 5 (auto-resume) detects the checkpoint automatically.

## Cell 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted at /content/drive')

## Cell 2 — Install Dependencies

Installs `uv` (fast pip replacement), then all required packages.  
Re-running this cell on resume is safe — everything is already cached.

In [ ]:
import subprocess, sys

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout[-2000:])
        print(result.stderr[-2000:])
        raise RuntimeError(f'Command failed: {cmd}')
    return result.stdout

# Install uv
print('Installing uv...')
run('curl -LsSf https://astral.sh/uv/install.sh | sh')
import os
os.environ['PATH'] = os.environ.get('HOME', '/root') + '/.local/bin:' + os.environ['PATH']
print('uv installed:', run('uv --version').strip())

# Core packages
packages = [
    'torchaudio',
    'pesq',
    'pystoi',
    'jiwer',
    'pythainlp',
    'matplotlib',
    'scipy',
    'tqdm',
]
print('Installing packages:', ', '.join(packages))
run(f'uv pip install --system {" ".join(packages)}')
print('All dependencies installed.')

## Cell 3 — Unzip Dataset to Local SSD

Reading `.npy` files from Drive is ~10× slower than from `/content/` (local NVMe).  
This cell copies `data_npy.zip` from Drive once and unpacks it.  
**Expected path on Drive**: `MyDrive/AI_builders/data_npy.zip`  
**Skip this cell** if already unzipped in this session (check the `Already unzipped` message).

In [ ]:
import os, zipfile, shutil, time

DRIVE_ZIP  = '/content/drive/MyDrive/AI_builders/data_npy.zip'
LOCAL_DIR  = '/content/data_local'
DATA_ROOT  = f'{LOCAL_DIR}/data_npy'

if os.path.isdir(DATA_ROOT):
    n_files = sum(len(fs) for _, _, fs in os.walk(DATA_ROOT))
    print(f'Already unzipped: {DATA_ROOT}  ({n_files:,} files) — skipping.')
else:
    if not os.path.isfile(DRIVE_ZIP):
        raise FileNotFoundError(
            f'data_npy.zip not found at {DRIVE_ZIP}\n'
            'Upload it to MyDrive/AI_builders/ first.'
        )
    os.makedirs(LOCAL_DIR, exist_ok=True)
    print(f'Unzipping {DRIVE_ZIP} → {LOCAL_DIR} ...')
    t0 = time.time()
    with zipfile.ZipFile(DRIVE_ZIP, 'r') as zf:
        zf.extractall(LOCAL_DIR)
    elapsed = time.time() - t0
    n_files = sum(len(fs) for _, _, fs in os.walk(DATA_ROOT))
    print(f'Done in {elapsed:.0f}s — {n_files:,} files at {DATA_ROOT}')

# Sanity-check expected splits
for split in ('train', 'val', 'test'):
    clean = os.path.join(DATA_ROOT, split, 'clean')
    noise = os.path.join(DATA_ROOT, split, 'noise')
    nc = len(os.listdir(clean)) if os.path.isdir(clean) else 0
    nn = len(os.listdir(noise)) if os.path.isdir(noise) else 0
    print(f'  {split:5s}: {nc:6,} clean  {nn:6,} noise')

print(f'\nDATA_ROOT = {DATA_ROOT}')

## Cell 4 — Copy Source Files from Drive

Copies the Python source files from `MyDrive/AI_builders/Colab_Ready_UNet/` to `/content/`  
so they run from local disk (faster imports, no Drive quota on reads).

In [ ]:
import shutil, os

DRIVE_SRC = '/content/drive/MyDrive/AI_builders/Colab_Ready_UNet'
LOCAL_DST = '/content'

FILES = [
    'model_unet_advanced.py',
    'train.py',
    'dataset.py',
    'losses.py',
    'evaluate.py',
    'inference.py',
]

if not os.path.isdir(DRIVE_SRC):
    raise FileNotFoundError(
        f'{DRIVE_SRC} not found.\n'
        'Upload the Colab_Ready_UNet folder to MyDrive/AI_builders/.'
    )

for f in FILES:
    src = os.path.join(DRIVE_SRC, f)
    dst = os.path.join(LOCAL_DST, f)
    shutil.copy2(src, dst)
    print(f'  {f}')

print('All source files ready in /content/')

## Cell 5 — Auto-Resume Check

Detects whether a previous checkpoint exists on Drive and sets `RESUME_FLAG` automatically.  
**Checkpoints are always saved to Drive** so they survive Colab session disconnects.

In [ ]:
import os

CKPT_DIR   = '/content/drive/MyDrive/AI_builders/checkpoints_unet'
BEST_CKPT  = os.path.join(CKPT_DIR, 'best.pt')

os.makedirs(CKPT_DIR, exist_ok=True)

if os.path.isfile(BEST_CKPT):
    size_mb = os.path.getsize(BEST_CKPT) / 1024 / 1024
    RESUME_FLAG = f'--resume {BEST_CKPT}'
    print(f'Checkpoint found ({size_mb:.1f} MB): {BEST_CKPT}')
    print(f'Auto-adding: {RESUME_FLAG}')
else:
    RESUME_FLAG = ''
    print('No checkpoint found — training will start from epoch 1.')

print(f'\nCKPT_DIR   = {CKPT_DIR}')
print(f'RESUME_FLAG = "{RESUME_FLAG}"')

## Cell 6 — Train

Key settings:
- `--epochs 30` — 30 epochs with `--steps_per_epoch 250` ≈ 4 000 clips/epoch; hard SNR [-5, 15] dB from epoch 1
- `--num_workers 2` — Colab-safe worker count (avoids IPC overhead)
- `--w_sisnr 1.0` — Direct SI-SDR focus; `best.pt` saved on Maximum Validation SI-SDR
- `--grad_clip 3.0` — Stabilises U-Net skip connections and deep conv layers
- `--amp_dtype bf16` — bfloat16 on A100/T4, avoids fp16 overflow on LF signals
- `--scheduler cosine` — smooth LR decay, no plateau collapse
- `--low_freq_boost 8.0` — 8x STFT loss weight for bins < 200 Hz
- `--lf_noise_ratio 0.70` — 70% LF + 30% general from categorised real noise pools
- `--patience 20` — allows 20 epochs without improvement before early stop
- `--ckpt_dir` points to Drive — checkpoints survive session restarts

In [ ]:
import subprocess, os

os.chdir('/content')

cmd = f"""
uv run python train.py \\
  --data_root      /content/data_local/data_npy \\
  --ckpt_dir       {CKPT_DIR} \\
  --epochs         30 \\
  --batch_size     16 \\
  --num_workers    2 \\
  --segment_seconds 2.0 \\
  --amp_dtype      bf16 \\
  --scheduler      cosine \\
  --lr             2e-4 \\
  --warmup_steps   500 \\
  --steps_per_epoch 250 \\
  --low_freq_boost 8.0 \\
  --lf_noise_ratio 0.70 \\
  --w_sisnr        1.0 \\
  --w_noise_stft   1.0 \\
  --grad_clip      3.0 \\
  --patience       20 \\
  --debug_every    5 \\
  {RESUME_FLAG}
""".strip()

print('Command:')
print(cmd)
print('\n' + '='*60 + '\n')

!{cmd}

## Cell 7 — Evaluate on Test Set

Runs MOS (DNSMOS → PESQ → SNR proxy), WER/CER (Whisper-small, Thai),  
and exports per-utterance CSV + spectrogram PNGs to Drive.

In [ ]:
import os
os.chdir('/content')

EVAL_OUT = f'{CKPT_DIR}/eval_outputs'

!uv run python evaluate.py \\
    --ckpt          {CKPT_DIR}/best.pt \\
    --data_root     /content/data_local/data_npy \\
    --split         test \\
    --max_files     200 \\
    --out_csv       {EVAL_OUT}/eval_per_utt.csv \\
    --out_summary   {EVAL_OUT}/eval_summary.csv \\
    --spec_dir      {EVAL_OUT}/spectrograms \\
    --language      thai

## Cell 8 — Audio Demo (2 Samples)

Enhances 2 test files and plays **Noisy Input / Enhanced Output / Clean Reference** for each.
Noisy input is synthesised as real LF + HF noise so the demo is self-contained
(no separate noisy test files required).

In [ ]:
import numpy as np
import torch
from IPython.display import Audio, display
from scipy.signal import lfilter
import glob, os

os.chdir('/content')
from inference import load_model, enhance_waveform

SR = 16_000
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = load_model(f'{CKPT_DIR}/best.pt', device)

test_files = sorted(glob.glob('/content/data_local/data_npy/test/clean/*.npy'))
if not test_files:
    raise RuntimeError('No test files found — check DATA_ROOT.')

rng = np.random.default_rng(42)

# Pink-noise IIR filter coefficients
_b = np.array([0.049922035, -0.095993537, 0.050612699, -0.004408786], dtype=np.float32)
_a = np.array([1, -2.494956002, 2.017265875, -0.522189400], dtype=np.float32)

for sample_idx in range(2):
    clean_np = np.load(test_files[sample_idx]).squeeze().astype(np.float32)

    # Simulate noisy: LF (pink) at -5 dB SNR + HF white at +10 dB SNR
    white = rng.standard_normal(len(clean_np)).astype(np.float32)
    lf_noise = lfilter(_b, _a, white).astype(np.float32)
    hf_noise = rng.standard_normal(len(clean_np)).astype(np.float32)

    clean_rms = float(np.sqrt(np.mean(clean_np**2))) + 1e-8
    alpha_lf  = clean_rms / (float(np.sqrt(np.mean(lf_noise**2))) + 1e-8) / 10**(5/20)
    alpha_hf  = clean_rms / (float(np.sqrt(np.mean(hf_noise**2))) + 1e-8) / 10**(10/20)

    noisy_np    = (clean_np + alpha_lf * lf_noise + alpha_hf * hf_noise).astype(np.float32)
    enhanced_np = enhance_waveform(model, noisy_np, device, SR)

    print('=' * 60)
    print(f'Sample {sample_idx + 1}: {os.path.basename(test_files[sample_idx])}')
    print('=' * 60)
    print('Noisy Input  (LF pink + HF white noise):')
    display(Audio(noisy_np, rate=SR))
    print('Enhanced Output  (model denoised):')
    display(Audio(enhanced_np, rate=SR))
    print('Clean Reference:')
    display(Audio(clean_np, rate=SR))
    print()


## Cell 9 — Training Curves & SI-SDR Progress

Plots **Training vs Validation Loss** and **Validation SI-SDR** over epochs.  
Reads `history.json` written by train.py into the checkpoint folder.  
Also saves `training_curves.png` to the same folder on Drive.

In [ ]:
import json, os
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

hist_path = os.path.join(CKPT_DIR, 'history.json')
if not os.path.isfile(hist_path):
    raise FileNotFoundError(
        f'history.json not found at {hist_path} \n'
        'Run Cell 6 (Train) first.'
    )

with open(hist_path) as f:
    history = json.load(f)

epochs       = [h['epoch']        for h in history]
train_loss   = [h['train_loss']   for h in history]
val_loss     = [h['val_loss']     for h in history]
val_sisnr_db = [h.get('val_sisnr_db') or float('nan') for h in history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('U-Net Training Progress', fontsize=14, fontweight='bold')

# --- Loss curves ---
ax1.plot(epochs, train_loss, label='Train Loss', color='#2196F3', linewidth=2)
ax1.plot(epochs, val_loss,   label='Val Loss',   color='#FF5722', linewidth=2)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Training vs Validation Loss')
ax1.legend(); ax1.grid(alpha=0.3)
ax1.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

# --- SI-SDR progress ---
valid_ep = [e for e, s in zip(epochs, val_sisnr_db) if not np.isnan(float(s or 'nan'))]
valid_si = [s for s in val_sisnr_db if not np.isnan(float(s or 'nan'))]
ax2.plot(valid_ep, valid_si, color='#4CAF50', linewidth=2, marker='o', markersize=4)
ax2.axhline(y=9.0, color='#FF9800', linestyle='--', linewidth=1.5, label='Target 9.0 dB')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('SI-SDR (dB)')
ax2.set_title('Validation SI-SDR Progress')
ax2.legend(); ax2.grid(alpha=0.3)
ax2.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

if valid_si:
    best_val = max(valid_si)
    best_ep  = valid_ep[valid_si.index(best_val)]
    ax2.annotate(
        f'Best: {best_val:.2f} dB',
        xy=(best_ep, best_val),
        xytext=(best_ep + max(1, len(epochs)//8), best_val - 0.6),
        arrowprops=dict(arrowstyle='->', color='black'),
        fontsize=9
    )

plt.tight_layout()
out_png = os.path.join(CKPT_DIR, 'training_curves.png')
plt.savefig(out_png, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {out_png}')
if valid_si:
    print(f'Best val SI-SDR: {max(valid_si):.2f} dB at epoch {valid_ep[valid_si.index(max(valid_si))]}')
